# 02 · Hiperparámetros con PSO
Wine = la tabla. SVM = el clasificador. **C y gamma = perillas del SVM** (no el alcohol).
PSO = el buscador. Grid search = el otro buscador (cuadrícula).

Aptitud que baja: `J = 1 - accuracy`.


In [ ]:
# Motor ABC + PSO (incluido para que Colab no dependa de rutas)
#!/usr/bin/env python3
"""Primitivas ABC y PSO usadas por los cuatro ejercicios."""

import numpy as np


def pso_minimize(objective, bounds, n_particles=20, iters=25, w=0.7, c1=1.5, c2=1.5, seed=42):
    """PSO global-best. Minimiza objective(x).

    Ciclo: representacion = posicion continua,
    inicializacion uniforme, aptitud = objective,
    comportamiento = inercia + pbest + gbest,
    evolucion por iteraciones, parada = iters.
    """
    rng = np.random.default_rng(seed)
    lo, hi = np.asarray(bounds[0], float), np.asarray(bounds[1], float)
    dim = lo.size
    pos = rng.uniform(lo, hi, size=(n_particles, dim))
    vel = np.zeros_like(pos)
    costs = np.array([objective(p) for p in pos])
    pbest, pbest_c = pos.copy(), costs.copy()
    g = int(np.argmin(pbest_c))
    gbest, gbest_c = pbest[g].copy(), float(pbest_c[g])
    hist = [gbest_c]
    swarm_hist = [pos.copy()]
    for _ in range(iters):
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = w * vel + c1 * r1 * (pbest - pos) + c2 * r2 * (gbest - pos)
        pos = np.clip(pos + vel, lo, hi)
        costs = np.array([objective(p) for p in pos])
        improved = costs < pbest_c
        pbest[improved] = pos[improved]
        pbest_c[improved] = costs[improved]
        g = int(np.argmin(pbest_c))
        if pbest_c[g] < gbest_c:
            gbest, gbest_c = pbest[g].copy(), float(pbest_c[g])
        hist.append(gbest_c)
        swarm_hist.append(pos.copy())
    return gbest, gbest_c, np.array(hist), swarm_hist


def abc_binary_maximize(objective, n_bits, n_bees=10, cycles=12, limit=4, seed=42):
    """ABC binario. Maximiza objective(bitstring).

    Ciclo: representacion = fuente de alimento binaria,
    inicializacion aleatoria, aptitud = objective,
    comportamiento = empleada / observadora / exploradora,
    evolucion por ciclos, parada = cycles.
    """
    rng = np.random.default_rng(seed)
    foods = rng.integers(0, 2, size=(n_bees, n_bits))
    empty = foods.sum(axis=1) == 0
    if empty.any():
        foods[empty, rng.integers(0, n_bits, size=int(empty.sum()))] = 1
    fit = np.array([objective(f) for f in foods])
    trials = np.zeros(n_bees, dtype=int)
    best_i = int(np.argmax(fit))
    best, best_f = foods[best_i].copy(), float(fit[best_i])
    hist = [best_f]

    def neighbor(src):
        k = int(rng.integers(0, n_bits))
        nxt = src.copy()
        nxt[k] ^= 1
        if nxt.sum() == 0:
            nxt[k] = 1
        return nxt

    for _ in range(cycles):
        for i in range(n_bees):
            cand = neighbor(foods[i])
            fc = objective(cand)
            if fc >= fit[i]:
                foods[i], fit[i], trials[i] = cand, fc, 0
            else:
                trials[i] += 1
        probs = fit - fit.min() + 1e-9
        probs = probs / probs.sum()
        for _o in range(n_bees):
            i = int(rng.choice(n_bees, p=probs))
            cand = neighbor(foods[i])
            fc = objective(cand)
            if fc >= fit[i]:
                foods[i], fit[i], trials[i] = cand, fc, 0
            else:
                trials[i] += 1
        for i in range(n_bees):
            if trials[i] >= limit:
                foods[i] = rng.integers(0, 2, size=n_bits)
                if foods[i].sum() == 0:
                    foods[i][int(rng.integers(0, n_bits))] = 1
                fit[i] = objective(foods[i])
                trials[i] = 0
        bi = int(np.argmax(fit))
        if fit[bi] > best_f:
            best, best_f = foods[bi].copy(), float(fit[bi])
        hist.append(best_f)
    return best, best_f, np.array(hist)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})

wine = load_wine()
X = StandardScaler().fit_transform(wine.data)
y = wine.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

def objective(p):
    C, g = 10 ** float(p[0]), 10 ** float(p[1])
    acc = cross_val_score(SVC(C=C, gamma=g, kernel="rbf"), Xtr, ytr, cv=inner, scoring="accuracy").mean()
    return float(1.0 - acc)   # 1 menos accuracy = lo que le falta para el 100%


In [ ]:
bounds = (np.array([-2.0, -4.0]), np.array([3.0, 1.0]))
best, cost, hist, swarm = pso_minimize(objective, bounds, n_particles=10, iters=10, seed=SEED)
C_star, g_star = 10 ** best[0], 10 ** best[1]
acc_pso_te = accuracy_score(yte, SVC(C=C_star, gamma=g_star, kernel="rbf").fit(Xtr, ytr).predict(Xte))

grid = GridSearchCV(SVC(kernel="rbf"),
    {"C": np.logspace(-2, 3, 5), "gamma": np.logspace(-4, 1, 5)},
    cv=inner, scoring="accuracy").fit(Xtr, ytr)
acc_grid_te = accuracy_score(yte, grid.predict(Xte))
acc_default = accuracy_score(yte, SVC().fit(Xtr, ytr).predict(Xte))

print(f"PSO     C={C_star:.3f}  gamma={g_star:.5f}  CV={1-cost:.4f}  test={acc_pso_te:.4f}")
print(f"Grid    C={grid.best_params_['C']:.3f}  gamma={grid.best_params_['gamma']:.5f}  test={acc_grid_te:.4f}")
print(f"Default test={acc_default:.4f}")


In [ ]:
Cs = np.linspace(-2, 3, 7); Gs = np.linspace(-4, 1, 7)
Z = np.zeros((len(Gs), len(Cs)))
for i, gv in enumerate(Gs):
    for j, cv_ in enumerate(Cs):
        Z[i, j] = objective([cv_, gv])

fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.contourf(Cs, Gs, Z, levels=12, cmap="viridis_r")
fig.colorbar(im, ax=ax, label="1 - accuracy  (error)")
last = swarm[-1]
ax.scatter(last[:,0], last[:,1], c="#f4d35e", s=28, edgecolor="k", label="enjambre")
ax.scatter(best[0], best[1], c="red", s=90, marker="*", label="PSO (estrella)")
ax.scatter(np.log10(grid.best_params_["C"]), np.log10(grid.best_params_["gamma"]),
           c="white", s=50, marker="D", edgecolor="k", label="Grid (rombo)")
ax.set_xlabel("log10 C"); ax.set_ylabel("log10 gamma")
ax.set_title("Tablero de perillas — no es el vino")
ax.legend(loc="lower left"); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.3))
vals = [acc_default, acc_grid_te, acc_pso_te]
bars = ax.bar(["SVM default", "Grid 25", "PSO"], vals, color=["#666","#4c78a8","#3ee0e8"])
ax.set_ylim(0.7, 1.05); ax.set_ylabel("accuracy test")
for b,v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.show()
